# DTCG Datacubes

In [1]:
import dtcg
import dtcg.integration.oggm_bindings as oggm_bindings
import oggm
from oggm.core import massbalance
import json
from pathlib import Path
from datetime import datetime, date, timezone, timedelta

import numpy as np
import xarray as xr


# Setup OGGM

In [2]:
base_url = "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands_w_data/W5E5/per_glacier/"
output_dir = Path("./static/data/datacube_gen/")

In [3]:
def load_ids_from_json(path: str) -> list:
    with open(path) as file:
        rgi_ids = json.load(file)
    assert isinstance(rgi_ids, list)
    return rgi_ids


iceland_ids = load_ids_from_json(path=output_dir / "vatnajokull_rgi_ids.json")
alpine_ids = load_ids_from_json(path=output_dir / "oetztal_rgi_ids.json")
rgi_ids = set(iceland_ids)
# assert len(rgi_ids) == len(iceland_ids + alpine_ids)

In [4]:
binder = dtcg.integration.oggm_bindings.BindingsCryotempo()

In [5]:
from oggm import workflow
from oggm.shop import w5e5


def get_data(rgi_ids: list):
    """Get dashboard data.

    Returns
    -------
    tuple
        Glacier directory, EOLIS-enhanced gridded data, and specific mass balance.
    """
    binder.init_oggm(dirname="gen-datacubes", reset=True)
    gdirs = binder.get_glacier_directories(
        rgi_ids=rgi_ids, from_prepro_level=4, prepro_border=80
    )
    print("Fetching OGGM data from shop...")

    binder.get_glacier_data(gdirs=gdirs)
    # workflow.execute_entity_task(
    #         gdirs=gdirs, task=w5e5.process_w5e5_data, daily=True
    #     )
    for gdir in gdirs:
        binder.set_flowlines(gdir)
    return gdirs

In [6]:
# rgi_ids = ["RGI60-06.00377"]
data = {}
gdirs = get_data(rgi_ids)
for gdir in gdirs:
    data[gdir.rgi_id] = {}

Fetching OGGM data from shop...


# L1 Datacubes

CryoSat-2 data if available from Specklia

In [ ]:
for gdir in gdirs:
    if "-06." in gdir.rgi_id:
        try:
            gdir, datacube = binder.get_eolis_data(gdir)
            data[gdir.rgi_id]["datacube"] = datacube
        except Exception as e:
            print(e)
            print(f"Failed to download Specklia for {gdir.rgi_id}.")
            data[gdir.rgi_id]["datacube"] = None
    else:
        data[gdir.rgi_id]["datacube"] = None
    data[gdir.rgi_id]["gdir"] = gdir

conflicting sizes for dimension 't': length 0 on the data but length 173 on coordinate 't'
Failed to download Specklia for RGI60-06.00404.


/home/nicolas/Documents/WORK/UoB/DTC/dtcg/dtcg/datacube/geozarr.py:199: UserWarning: Metadata mapping is missing for the following variables: ['eolis_elevation_change_sigma_timeseries', 'eolis_elevation_change_timeseries', 'eolis_gridded_elevation_change', 'eolis_gridded_elevation_change_sigma']. The metadata for these variables might not be compliant with Climate and Forecast conventions https://cfconventions.org/.
  ds = self.update_metadata(ds, ds_name)
/home/nicolas/Documents/WORK/UoB/DTC/dtcg/dtcg/datacube/geozarr.py:199: UserWarning: Metadata mapping is missing for the following variables: ['eolis_elevation_change_sigma_timeseries', 'eolis_elevation_change_timeseries', 'eolis_gridded_elevation_change', 'eolis_gridded_elevation_change_sigma']. The metadata for these variables might not be compliant with Climate and Forecast conventions https://cfconventions.org/.
  ds = self.update_metadata(ds, ds_name)
/home/nicolas/Documents/WORK/UoB/DTC/dtcg/dtcg/datacube/geozarr.py:199: UserWa

# L2 Datacubes

SMB and monthly runoff for:
- Daily_Hugonnet
- Daily_SfcTracking_Hugonnet
- Daily_CryoSat (if available)
- Daily_SfcTracking_CryoSat (if available)

## Calibrate Model

In [ ]:
binder.calibrator.model_matrix = {}

In [ ]:
# Hugonnet
for glacier_id, glacier in data.items():
    gdir = glacier["gdir"]
    for key in ["smb", "model"]:
        if key not in data[glacier_id].keys():
            glacier[key] = {}
    ref_mb = binder.calibrator.get_geodetic_mb(gdir=gdir, dataset=None)
    source = "Hugonnet"
    geo_period = "2010-01-01_2020-01-01"
    for oggm_model in [massbalance.DailyTIModel]:  # , massbalance.SfcTypeTIModel]:
        if issubclass(oggm_model, massbalance.SfcTypeTIModel):
            sfc_model_kwargs = {
                "climate_resolution": "daily",
            }
        else:
            sfc_model_kwargs = {}
        binder.calibrator.set_model_matrix(
            name=f"{oggm_model.__name__}_{source}",
            model=oggm_model,
            geo_period=geo_period,
            daily=True,
            source=source,
            **sfc_model_kwargs,
        )
    mb_model_calib, mb_model_flowlines, smb = binder.calibrator.calibrate(
        model_matrix=binder.calibrator.model_matrix,
        gdir=gdir,
        ref_mb=ref_mb,
        # **sfc_model_kwargs,
    )

    glacier["smb"][source] = smb
    glacier["model"][source] = mb_model_flowlines
    binder.calibrator.model_matrix = {}

100%|██████████| 1/1 [00:00<00:00,  1.42it/s]


KeyError: 'gdir'

In [ ]:
for glacier_id, glacier in data.items():
    gdir = glacier["gdir"]
    for key in ["smb", "model"]:
        if key not in data[glacier_id].keys():
            glacier[key] = {}
    if glacier["datacube"] is not None:
        ref_mb = binder.calibrator.get_geodetic_mb(
            gdir=gdir, dataset=glacier["datacube"].get_layer("L1")
        )
        source = "CryoTEMPO-EOLIS"
        geo_period = "2011-01-01_2020-01-01"
        for oggm_model in [massbalance.DailyTIModel]:  # , massbalance.SfcTypeTIModel]:
            if issubclass(oggm_model, massbalance.SfcTypeTIModel):
                sfc_model_kwargs = {
                    "climate_resolution": "daily",
                }
            else:
                sfc_model_kwargs = {}

            binder.calibrator.set_model_matrix(
                name=f"{oggm_model.__name__}_{source}",
                model=oggm_model,
                geo_period=geo_period,
                daily=True,
                source=source,
                **sfc_model_kwargs,
            )
            # print(binder.calibrator.model_matrix)
        mb_model_calib, mb_model_flowlines, smb = binder.calibrator.calibrate(
            model_matrix=binder.calibrator.model_matrix,
            gdir=gdir,
            ref_mb=ref_mb,
            # **sfc_model_kwargs,
        )

        glacier["smb"][source] = smb
        glacier["model"][source] = mb_model_flowlines
        binder.calibrator.model_matrix = {}
    binder.calibrator.model_matrix = {}

100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


KeyError: 'gdir'

In [ ]:
data["RGI60-06.00377"]["smb"]

{'Hugonnet': {'DailyTIModel_Hugonnet_2010-01-01_2020-01-01': array([ 6.50677933,  0.        ,  0.        , ..., 24.33497627,
          0.        ,  4.23603162], shape=(7305,))},
 'CryoTEMPO-EOLIS': {'DailyTIModel_CryoTEMPO-EOLIS_2011-01-01_2020-01-01': array([ 6.50677933,  0.        ,  0.        , ..., 24.33497627,
          0.        ,  5.15768611], shape=(7305,))}}

In [ ]:
def get_time_range(y0=2000, y1=2020):
    start_dt = datetime(2000, 1, 1, tzinfo=timezone.utc)
    end_dt = datetime(2020, 1, 1, tzinfo=timezone.utc)
    delta = timedelta(days=1)
    dates = []

    while start_dt < end_dt:
        # add current date to list by converting  it to iso format
        dates.append(start_dt.timestamp())
        # increment start date by timedelta
        start_dt += delta

    return dates

In [ ]:
hugonnet_citation = "Hugonnet, R., McNabb, R., Berthier, E. et al. Accelerated global glacier mass loss in the early twenty-first century. Nature 592, 726-731 (2021). https://doi.org/10.1038/s41586-021-03436-z"


# construct an xarray dataset with the modelled mass balance to add to the datacube
# years = oggm.utils.float_years_timeseries(y0=2000,y1=2019, include_last_year=True,
# monthly=False)
year_dts = get_time_range(y0=2000, y1=2020)
data_arrs = []
for glacier_id, glacier in data.items():
    if glacier["datacube"] is not None:
        eolis_citation = (
            glacier["datacube"]
            .get_layer("L1")
            .eolis_elevation_change_timeseries.attrs.get("references", "")
        )
        for _, source in glacier["smb"].items():
            for label, model_output in source.items():
                input_data_citation = (
                    hugonnet_citation if "Hugonnet" in label else eolis_citation
                )
                data_arrs.append(
                    xr.DataArray(
                        model_output,
                        dims=("t"),
                        coords={"t": year_dts},
                        name=label,
                        attrs={
                            "institution": "OGGM / DTC-Glaciers",
                            "standard_name": "land_ice_surface_specific_mass_balance",
                            "long_name": "Modelled land ice surface specific mass balance",
                            "references": "Maussion, F., Butenko, A., Champollion, N., Dusch, M., Eis, J., Fourteau, K., Gregor, P., Jarosch, A. H., Landmann, J., Oesterle, F., Recinos, B., Rothenpieler, T., Vlug, A., Wild, C. T., and Marzeion, B.: The Open Global Glacier Model (OGGM) v1.1, Geosci. Model Dev., 12, 909-931, https://doi.org/10.5194/gmd-12-909-2019, 2019. \n "
                            + input_data_citation,
                            "source": "OGGM modelled Specific Mass Balance informed by satellite observations.",
                            "units": "mm w.e.",
                            "comment": "N/A",
                        },
                    )
                )
        ds = xr.merge(data_arrs)
        ds.attrs.clear()  # clear dataset level attributes

        # add the new dataset as L2 layer to the datacube
        glacier["datacube"].add_layer(ds, "L2", overwrite=True)

/home/nicolas/Documents/WORK/UoB/DTC/dtcg/dtcg/datacube/geozarr.py:199: UserWarning: Metadata mapping is missing for the following variables: ['DailyTIModel_CryoTEMPO-EOLIS_2011-01-01_2020-01-01', 'DailyTIModel_Hugonnet_2010-01-01_2020-01-01']. The metadata for these variables might not be compliant with Climate and Forecast conventions https://cfconventions.org/.
  ds = self.update_metadata(ds, ds_name)


MergeError: conflicting values for variable 'DailyTIModel_Hugonnet_2010-01-01_2020-01-01' on objects to be combined. You can skip this check by specifying compat='override'.

### Export Zarr

In [ ]:
for glacier_id, glacier in data.items():
    if glacier["datacube"] is not None:
        output_path = Path(output_dir / glacier_id / f"{glacier_id}.zarr")
        glacier["datacube"].export(output_path)
        print(f"GeoZarr exported to: {output_path}")

GeoZarr exported to: static/data/datacube_gen/RGI60-06.00377/RGI60-06.00377.zarr


# Precomputed data
- gdir.json
- outlines.shp

In [ ]:
def export_gdir(data, save_path):
    gdir = data["gdir"]

    gdir_attrs = {
        "rgi_id": gdir.rgi_id,
        "name": gdir.name,
        "glims_id": gdir.glims_id,
        "rgi_region": gdir.rgi_region,
        "rgi_region_name": gdir.rgi_region_name,
        "rgi_subregion": gdir.rgi_subregion,
        "rgi_subregion_name": gdir.rgi_subregion_name,
        "cenlat": gdir.cenlat,
        "cenlon": gdir.cenlon,
    }

    with open(save_path / "gdir.json", mode="w", encoding="utf-8") as file:
        json.dump(gdir_attrs, file)


def export_runoff(data, save_path):
    data["runoff_data"]["monthly_runoff"].to_netcdf(save_path / "runoff.nc", mode="w")


def export_smb(data: dict, save_path):
    np.savez(file=save_path / "smb.npz", **data)


def export_outlines(data: dict, save_path):
    glacier_outlines_gdf = data["gdir"].read_shapefile("outlines")
    glacier_outlines_gdf.to_feather(path=save_path / "outlines.shp")

def set_save_folder(rgi_id, folder="../ext/data/l2_precompute/"):
    save_folder = Path(folder)
    assert save_folder.is_dir()
    save_path = save_folder / f"{rgi_id}/"
    Path(save_path).mkdir(parents=True, exist_ok=True)
    assert save_path.is_dir()
    return save_path

def export_eolis_data(data, save_path):
    eolis_data = data["datacube"].get_layer("L1")[["eolis_elevation_change_timeseries","eolis_elevation_change_sigma_timeseries"]]
    eolis_data.to_netcdf(save_path/"eolis.nc",mode="w",)

for glacier_id, glacier in data.items():
    output_path = set_save_folder(rgi_id=glacier_id, folder=output_dir)
    export_gdir(data=glacier, save_path=output_path)
    if glacier["datacube"] is not None:
        export_eolis_data(data=glacier, save_path=output_path)
        smb = glacier["smb"]["Hugonnet"] | glacier["smb"]["CryoTEMPO-EOLIS"]
        
    else:
        smb = glacier["smb"]["Hugonnet"]
    export_smb(data=smb, save_path=output_path)
    glacier["runoff_data"] = binder.get_aggregate_runoff(gdir=glacier["gdir"])
    export_runoff(data=glacier, save_path=output_path)
    export_outlines(data=glacier, save_path=output_path)